# ARGO Phase 1: Dataset and Expert-Disagreement Audit

**Purpose:** Reproduce the ARGO dataset structure, verify consensus counts, quantify disagreement among the three electrophysiologists, and create the analysis table needed for the planned disagreement-aware modeling study.

This notebook intentionally performs **no train/test modeling yet**. Its job is to establish the exact sample structure and prevent leakage before models are fit.

Expected public dataset:
- 1,962 electrograms
- 9 patients
- Consensus labels: 940 AVP, 776 Physiological, 246 Unknown
- Three independent expert annotations plus final consensus for every record


In [ ]:
# Run this in Colab or another Python environment.
# Uncomment only if packages are missing.
# %pip -q install wfdb pandas numpy scipy scikit-learn matplotlib

from pathlib import Path
from collections import Counter
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
import wfdb

pd.set_option("display.max_columns", 100)


## 1. Point the notebook at ARGO

Either:
1. Download the ZIP from PhysioNet and unzip it, then set `ROOT`, or
2. In Colab, the optional commands below can download and unzip the public release.

The expected directory contains `ARGODataset_Folder/Pt1` through `Pt9`.


In [ ]:
# OPTIONAL COLAB DOWNLOAD:
# !wget -q --show-progress -O /content/argo.zip "https://physionet.org/content/argo/get-zip/1.0.0/"
# !unzip -q /content/argo.zip -d /content/argo

# Change this path if needed:
CANDIDATES = [
    Path("/content/argo"),
    Path("/content"),
    Path("."),
]

def find_argo_root(candidates):
    for base in candidates:
        hits = list(base.rglob("ARGODataset_Folder")) if base.exists() else []
        if hits:
            return hits[0].parent
    raise FileNotFoundError(
        "Could not find ARGODataset_Folder. Unzip ARGO and update CANDIDATES."
    )

ROOT = find_argo_root(CANDIDATES)
DATA = ROOT / "ARGODataset_Folder"
print("ARGO root:", ROOT.resolve())
print("Data folder:", DATA.resolve())


## 2. Verify patient metadata and record counts

In [ ]:
meta_path = ROOT / "Additional_subject_data.csv"
meta = pd.read_csv(meta_path, sep=";")
meta.columns = [c.strip() for c in meta.columns]
display(meta)

patients = sorted([p for p in DATA.glob("Pt*") if p.is_dir()],
                  key=lambda p: int(p.name[2:]))

record_rows = []
for p in patients:
    headers = sorted(p.glob("P*.hea"))
    for h in headers:
        record_rows.append({"patient": p.name, "record": h.stem, "base": str(h.with_suffix(""))})

records = pd.DataFrame(record_rows)
counts = records.groupby("patient").size().rename("records").reset_index()
display(counts)

assert len(records) == 1962, f"Expected 1962 records, found {len(records)}"
assert len(patients) == 9, f"Expected 9 patients, found {len(patients)}"
print("PASS: 1,962 records across 9 patients.")


## 3. Annotation parser

ARGO stores four WFDB annotation files per record:
- `annotation_ann1`
- `annotation_ann2`
- `annotation_ann3`
- `annotation_consensus`

The dataset documentation states that class is one of **A / P / U**, and AVP onset/end are indicated by `(` and `)` sample markers.

The parser below deliberately checks both `symbol` and `aux_note` so it remains robust to how the class token is encoded by WFDB.


In [ ]:
CLASS_TOKENS = {"A", "P", "U"}

def parse_annotation(record_base, extension):
    ann = wfdb.rdann(record_base, extension)
    symbols = [str(x).strip() for x in getattr(ann, "symbol", [])]
    aux = [str(x).strip() for x in getattr(ann, "aux_note", [])]
    samples = np.asarray(getattr(ann, "sample", []), dtype=int)

    # Search aux notes first, then annotation symbols.
    label = None
    for seq in (aux, symbols):
        for token in seq:
            cleaned = token.strip().strip("\x00").strip()
            if cleaned in CLASS_TOKENS:
                label = cleaned
                break
            # Handles strings such as "(A" or "A " without guessing from words.
            hits = [c for c in CLASS_TOKENS if cleaned == c]
            if hits:
                label = hits[0]
                break
        if label is not None:
            break

    onset = None
    end = None
    for s, sym in zip(samples, symbols):
        if sym == "(":
            onset = int(s)
        elif sym == ")":
            end = int(s)

    return {
        "label": label,
        "onset": onset,
        "end": end,
        "symbols": symbols,
        "aux_note": aux,
        "samples": samples.tolist(),
    }

# Diagnostic on the first record:
first = records.iloc[0]
print("Diagnostic record:", first["patient"], first["record"])
for ext in ["annotation_ann1", "annotation_ann2", "annotation_ann3", "annotation_consensus"]:
    print(ext, parse_annotation(first["base"], ext))


### If the diagnostic shows `label=None`

Stop there and inspect the printed `symbols` / `aux_note`. The rest of the notebook should not be run until the parser is adapted to the exact token representation. This guard prevents silently mislabeling the data.


In [ ]:
# Parse all four annotations.
rows = []
extensions = {
    "ann1": "annotation_ann1",
    "ann2": "annotation_ann2",
    "ann3": "annotation_ann3",
    "consensus": "annotation_consensus",
}

for _, r in records.iterrows():
    row = {"patient": r["patient"], "record": r["record"], "base": r["base"]}
    for short, ext in extensions.items():
        a = parse_annotation(r["base"], ext)
        row[f"{short}_label"] = a["label"]
        row[f"{short}_onset"] = a["onset"]
        row[f"{short}_end"] = a["end"]
    rows.append(row)

ann_df = pd.DataFrame(rows)

label_cols = ["ann1_label", "ann2_label", "ann3_label", "consensus_label"]
missing = ann_df[label_cols].isna().sum()
print("Missing parsed class labels:")
print(missing)
assert missing.sum() == 0, "Parser did not recover all class labels."
display(ann_df.head())


## 4. Reproduce consensus class counts

In [ ]:
consensus_counts = ann_df["consensus_label"].value_counts().reindex(["A","P","U"])
print(consensus_counts)

EXPECTED = pd.Series({"A": 940, "P": 776, "U": 246})
assert consensus_counts.equals(EXPECTED), (
    f"Consensus count mismatch. Expected {EXPECTED.to_dict()}, got {consensus_counts.to_dict()}"
)
print("PASS: published consensus totals reproduced exactly.")


In [ ]:
patient_consensus = (
    pd.crosstab(ann_df["patient"], ann_df["consensus_label"])
    .reindex(columns=["A","P","U"], fill_value=0)
)
patient_consensus["Total"] = patient_consensus.sum(axis=1)
for c in ["A","P","U"]:
    patient_consensus[f"{c}_pct"] = 100 * patient_consensus[c] / patient_consensus["Total"]

display(patient_consensus.round(2))


## 5. Quantify raw expert disagreement

Primary disagreement variables:
- `n_unique`: number of distinct labels among the 3 experts
- `all_agree`: all 3 experts gave the same label
- `two_one_split`: exactly 2 experts agree
- `three_way_split`: A, P, and U all appear
- `expert_entropy`: normalized entropy of the 3-vote distribution, in [0, 1]
- `majority_label`: hard majority when one exists
- `consensus_equals_majority`: whether discussion-based consensus simply matches majority vote


In [ ]:
def normalized_vote_entropy(labels):
    counts = Counter(labels)
    probs = np.array(list(counts.values()), dtype=float) / 3.0
    h = -(probs * np.log(probs)).sum()
    return float(h / np.log(3.0))

def majority_label(labels):
    c = Counter(labels)
    lab, n = c.most_common(1)[0]
    return lab if n >= 2 else None

expert_cols = ["ann1_label","ann2_label","ann3_label"]

ann_df["n_unique"] = ann_df[expert_cols].nunique(axis=1)
ann_df["all_agree"] = ann_df["n_unique"].eq(1)
ann_df["two_one_split"] = ann_df["n_unique"].eq(2)
ann_df["three_way_split"] = ann_df["n_unique"].eq(3)
ann_df["expert_entropy"] = ann_df[expert_cols].apply(
    lambda r: normalized_vote_entropy(r.tolist()), axis=1
)
ann_df["majority_label"] = ann_df[expert_cols].apply(
    lambda r: majority_label(r.tolist()), axis=1
)
ann_df["consensus_equals_majority"] = (
    ann_df["majority_label"].notna()
    & ann_df["majority_label"].eq(ann_df["consensus_label"])
)

summary = pd.Series({
    "N": len(ann_df),
    "all_three_agree_n": int(ann_df["all_agree"].sum()),
    "all_three_agree_pct": 100 * ann_df["all_agree"].mean(),
    "two_one_split_n": int(ann_df["two_one_split"].sum()),
    "two_one_split_pct": 100 * ann_df["two_one_split"].mean(),
    "three_way_split_n": int(ann_df["three_way_split"].sum()),
    "three_way_split_pct": 100 * ann_df["three_way_split"].mean(),
    "consensus_differs_from_majority_n": int(
        (ann_df["majority_label"].notna() & ~ann_df["consensus_equals_majority"]).sum()
    ),
})
display(summary.to_frame("value"))


## 6. Reproduce inter-rater agreement

Fleiss' kappa is computed directly from the 3 expert labels. Cohen's kappa is then calculated for each expert versus consensus.


In [ ]:
def fleiss_kappa_from_labels(frame, cols, categories=("A","P","U")):
    mat = np.zeros((len(frame), len(categories)), dtype=int)
    cat_to_i = {c:i for i,c in enumerate(categories)}
    for i, row in enumerate(frame[cols].itertuples(index=False, name=None)):
        for lab in row:
            mat[i, cat_to_i[lab]] += 1

    n = len(cols)
    P_i = (np.sum(mat**2, axis=1) - n) / (n * (n - 1))
    P_bar = P_i.mean()
    p_j = mat.sum(axis=0) / (len(frame) * n)
    P_e = np.sum(p_j**2)
    return (P_bar - P_e) / (1 - P_e)

fk = fleiss_kappa_from_labels(ann_df, expert_cols)
print("Fleiss kappa, 3 independent annotators:", round(fk, 4))
print("Published reference: ~0.61")

for c in expert_cols:
    k = cohen_kappa_score(ann_df[c], ann_df["consensus_label"], labels=["A","P","U"])
    print(f"{c} vs consensus: {k:.4f}")

assert abs(fk - 0.61) < 0.03, "Fleiss kappa is unexpectedly far from the published value."


## 7. Disagreement by consensus class and patient

This is central to the paper because disagreement may be highly concentrated in particular patients or in the Unknown class. If so, later models must demonstrate that uncertainty is not simply learning patient identity.


In [ ]:
by_class = (
    ann_df.groupby("consensus_label")
    .agg(
        n=("record","size"),
        all_agree_pct=("all_agree", lambda x: 100*x.mean()),
        two_one_pct=("two_one_split", lambda x: 100*x.mean()),
        three_way_pct=("three_way_split", lambda x: 100*x.mean()),
        mean_expert_entropy=("expert_entropy","mean"),
    )
)
display(by_class.round(3))

by_patient = (
    ann_df.groupby("patient")
    .agg(
        n=("record","size"),
        all_agree_pct=("all_agree", lambda x: 100*x.mean()),
        mean_expert_entropy=("expert_entropy","mean"),
        consensus_majority_match_pct=("consensus_equals_majority", lambda x: 100*x.mean()),
    )
)
display(by_patient.round(3))


## 8. AVP delineation audit

For each expert and consensus:
- Count usable AVP onset/end pairs
- Compute AVP duration
- Identify cases where all 3 experts call AVP, enabling direct inter-rater boundary comparison


In [ ]:
for who in ["ann1","ann2","ann3","consensus"]:
    is_avp = ann_df[f"{who}_label"].eq("A")
    usable = is_avp & ann_df[f"{who}_onset"].notna() & ann_df[f"{who}_end"].notna()
    durations = ann_df.loc[usable, f"{who}_end"] - ann_df.loc[usable, f"{who}_onset"]
    print(
        who,
        "AVP labels =", int(is_avp.sum()),
        "| usable delineations =", int(usable.sum()),
        "| median duration samples/ms =", float(durations.median()) if len(durations) else None
    )

all3_avp = ann_df[expert_cols].eq("A").all(axis=1)
print("\nAll three independent experts labeled AVP:", int(all3_avp.sum()))


## 9. Save the Phase-1 audit outputs

In [ ]:
OUT = Path("argo_audit_outputs")
OUT.mkdir(exist_ok=True)

ann_df.to_csv(OUT / "argo_annotation_audit.csv", index=False)
patient_consensus.to_csv(OUT / "consensus_counts_by_patient.csv")
by_class.to_csv(OUT / "disagreement_by_consensus_class.csv")
by_patient.to_csv(OUT / "disagreement_by_patient.csv")

# Patient-level class distribution figure
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = patient_consensus[["A","P","U"]]
plot_df.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("ARGO consensus label distribution by patient")
ax.set_xlabel("Patient")
ax.set_ylabel("Number of EGMs")
fig.tight_layout()
fig.savefig(OUT / "consensus_label_distribution_by_patient.png", dpi=200)
plt.show()

print("Saved outputs to:", OUT.resolve())


## Phase-1 decision gates before modeling

Proceed to model development only if:

1. The dataset reproduces **1,962** records and **940 / 776 / 246** consensus A/P/U counts.
2. The parser recovers all four label streams without missing values.
3. Fleiss' κ is close to the published ~0.61.
4. We quantify how many records are unanimous, 2–1 splits, and three-way splits.
5. We quantify how often consensus differs from simple majority vote.
6. We verify usable AVP onset/end annotations.
7. We inspect disagreement by patient to ensure uncertainty modeling is not a patient-ID shortcut.

### Locked evaluation principle for later phases

**No random record-level cross-validation.**  
Primary evaluation will be **leave-one-patient-out (LOPO)**, with both:
- pooled record-level performance, and
- macro-averaged patient-level performance.

This is necessary because Pt4 and Pt6 dominate different label classes.
